# Predicción de rotura de stock - TFM Altadis## Objetivo 4: Predicción de rupturas de stockEste notebook entrena y compara tres modelos de clasificación binaria para predecir si un registro de venta (combinación día-outlet-producto) presentará rotura de stock declarada (Flag_OoS = 1) o no (Flag_OoS = 0):1. **Regresión Logística** — modelo base, interpretable.2. **Árbol de Decisión** — interpretable, permite visualizar las reglas de decisión.3. **XGBoost** — boosting de árboles, normalmente el más preciso.La variable objetivo presenta un desequilibrio extremo: solo 757 casos positivos sobre 950.217 registros (0,08%), muy superior al desequilibrio moderado del ejemplo del Titanic visto en clase. Se aplican las mismas técnicas de balanceo (`class_weight='balanced'`, `scale_pos_weight`) vistas en las Clases 12 y 13.**Variable objetivo:** `Flag_OoS`**Variables predictoras:** `Es_Festivoo`, `Flag_Ruta`, `Delivery_Unidades`, `Dia_Semana`, `Es_Fin_Semana`, `Tam_m2`, `Renta_Media_Provincial`, `Format`Nota: `Sales_Uds` no se incluye como predictor, ya que la rotura de stock es causa de una venta baja o nula, no al revés; incluirla introduciría fuga de información (*data leakage*).**Fichero de entrada esperado:** `TablaModPredictivoOutOfStock.csv` (separador `;`, sin cabecera).

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifier, plot_treefrom sklearn.metrics import (accuracy_score, confusion_matrix,                              classification_report, precision_score,                              recall_score, f1_score)from xgboost import XGBClassifier

## 1. Carga de datos

In [ ]:
cols = ['Affiliated_Code','Product_Code','Sales_DAY','Flag_OoS','Es_Festivoo',        'Flag_Ruta','Delivery_Unidades','Dia_Semana','Es_Fin_Semana',        'Tam_m2','Renta_Media_Provincial','Format']df = pd.read_csv('TablaModPredictivoOutOfStock.csv', sep=';',                  header=None, names=cols, encoding='utf-8-sig')print(df.shape)df.head()

In [ ]:
print(df.isnull().sum())print()print('Distribución de la variable objetivo:')print(df['Flag_OoS'].value_counts())print((df['Flag_OoS'].value_counts(normalize=True) * 100).round(3))

## 2. EDAComparamos la tasa de rotura según las variables de contexto, antes de entrenar ningún modelo, para tener una primera intuición de qué factores se asocian con la rotura de stock.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))# Rotura por festivodf.groupby('Es_Festivoo')['Flag_OoS'].mean().plot.bar(    ax=axes[0], color=['steelblue','salmon'],    title='Tasa de rotura: festivo vs. no festivo')axes[0].set_ylabel('Tasa de rotura')axes[0].set_xticklabels(['No festivo','Festivo'], rotation=0)# Rotura por día de la semanadf.groupby('Dia_Semana')['Flag_OoS'].mean().sort_values().plot.bar(    ax=axes[1], color='steelblue',    title='Tasa de rotura por día de la semana')axes[1].set_ylabel('Tasa de rotura')plt.tight_layout()plt.savefig('OoS_EDA_Contexto.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

## 3. PreprocesadoSe codifica `Tam_m2` como variable ordinal numérica, siguiendo el mismo criterio ya documentado y justificado en el análisis de clustering (punto medio de cada intervalo, con la aproximación declarada de 26 para la categoría ambigua `>20m2`).Las variables categóricas restantes (`Dia_Semana`, `Format`) se codifican mediante one-hot encoding, igual que en la Clase 13.

In [ ]:
# ── 1. Separar target y hacer copia ───────────────────────────────────df_prep = df.copy()y = df_prep['Flag_OoS']# ── 2. Codificación ordinal de Tam_m2 (mismo criterio que en el clustering) ──mapa_tam = {    '<2m2': 1, '2-5m2': 3.5, '5-10m2': 7.5, '10-20m2': 15,    '>20m2': 26, '20-30m2': 25, '>30m2': 35}df_prep['Tam_m2_Numerico'] = df_prep['Tam_m2'].map(mapa_tam)# ── 3. Variables predictoras ──────────────────────────────────────────X = df_prep[['Es_Festivoo','Flag_Ruta','Delivery_Unidades','Es_Fin_Semana',             'Tam_m2_Numerico','Renta_Media_Provincial','Dia_Semana','Format']]# ── 4. One-hot encoding de variables categóricas ──────────────────────X = pd.get_dummies(X, columns=['Dia_Semana','Format'], drop_first=True)print(X.shape)X.head()

## 4. División train/testSe utiliza `stratify=y` para garantizar la misma proporción de roturas en entrenamiento y test, igual que en la Clase 12. Dado el desequilibrio extremo (0,08% de positivos), es especialmente importante mantener esta proporción en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y,    test_size=0.20,    random_state=42,    stratify=y)print(f'Train: {X_train.shape[0]} registros | Test: {X_test.shape[0]} registros')print()print('Proporción de roturas en train:', y_train.mean().round(5))print('Proporción de roturas en test:', y_test.mean().round(5))

## 5. Regresión Logística (balanceada)

In [ ]:
modelo_log = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)modelo_log.fit(X_train, y_train)y_pred_log = modelo_log.predict(X_test)y_prob_log = modelo_log.predict_proba(X_test)[:, 1]print('── Regresión Logística ──')print(classification_report(y_test, y_pred_log, target_names=['Sin rotura','Con rotura']))

## 6. Árbol de DecisiónIgual que en la Clase 13: primero un árbol sin podar para diagnosticar sobreajuste, después un árbol podado y balanceado.

In [ ]:
# Árbol sin podar: diagnóstico de sobreajustearbol_full = DecisionTreeClassifier(random_state=42)arbol_full.fit(X_train, y_train)acc_train = accuracy_score(y_train, arbol_full.predict(X_train))acc_test = accuracy_score(y_test, arbol_full.predict(X_test))print(f'Accuracy train: {acc_train:.4f} | Accuracy test: {acc_test:.4f}')print('Si la diferencia es grande, hay sobreajuste.')

In [ ]:
# Árbol podado y balanceadoarbol = DecisionTreeClassifier(    max_depth=4,    min_samples_leaf=20,    class_weight='balanced',    random_state=42)arbol.fit(X_train, y_train)y_pred_arbol = arbol.predict(X_test)print('── Árbol de Decisión (podado, balanceado) ──')print(classification_report(y_test, y_pred_arbol, target_names=['Sin rotura','Con rotura']))

In [ ]:
plt.figure(figsize=(22, 10))plot_tree(    arbol,    feature_names=X.columns,    class_names=['Sin rotura','Con rotura'],    filled=True,    proportion=True,    rounded=True,    fontsize=8)plt.savefig('OoS_Arbol_Decision.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
importancias = pd.DataFrame({    'variable': X.columns,    'importancia': arbol.feature_importances_}).sort_values('importancia', ascending=True)fig, ax = plt.subplots(figsize=(8, 6))ax.barh(importancias['variable'], importancias['importancia'], color='steelblue')ax.set_xlabel('Importancia (reducción de impureza Gini)')ax.set_title('Importancia de variables - Árbol de Decisión')plt.tight_layout()plt.savefig('OoS_Importancia_Variables.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

## 7. XGBoost (balanceado)`scale_pos_weight` es el equivalente en XGBoost a `class_weight='balanced'`. Se calcula como la proporción entre clase negativa y positiva.

In [ ]:
peso_positivo = (y_train == 0).sum() / (y_train == 1).sum()print('scale_pos_weight calculado:', round(peso_positivo, 1))xgb_modelo = XGBClassifier(    max_depth=4,    learning_rate=0.1,    n_estimators=200,    scale_pos_weight=peso_positivo,    random_state=42,    eval_metric='logloss')xgb_modelo.fit(X_train, y_train)y_pred_xgb = xgb_modelo.predict(X_test)print('── XGBoost (balanceado) ──')print(classification_report(y_test, y_pred_xgb, target_names=['Sin rotura','Con rotura']))

## 8. Comparación final de los tres modelos

In [ ]:
def resumen_metricas(y_true, y_pred, nombre):    return {        'Modelo': nombre,        'Accuracy': accuracy_score(y_true, y_pred),        'Precision': precision_score(y_true, y_pred),        'Recall': recall_score(y_true, y_pred),        'F1': f1_score(y_true, y_pred)    }tabla = pd.DataFrame([    resumen_metricas(y_test, y_pred_log, 'Regresión Logística'),    resumen_metricas(y_test, y_pred_arbol, 'Árbol de Decisión'),    resumen_metricas(y_test, y_pred_xgb, 'XGBoost'),]).set_index('Modelo')print(tabla.round(4))

In [ ]:
tabla.plot.bar(figsize=(10, 5), rot=0,               color=['#3498DB','#2ECC71','#F39C12','#9B59B6'])plt.title('Comparación de modelos — Predicción de rotura de stock', fontsize=13, fontweight='bold')plt.ylabel('Valor de la métrica')plt.ylim(0, 1)plt.legend(loc='lower right')plt.tight_layout()plt.savefig('OoS_Comparacion_Modelos.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

## 9. Interpretación de resultadosDado el desequilibrio extremo de la variable objetivo (0,08% de casos positivos), se espera que el Accuracy sea elevado en los tres modelos casi por defecto (un modelo que prediga siempre "sin rotura" ya acertaría en el 99,92% de los casos), por lo que la métrica relevante para evaluar la utilidad real del modelo es el Recall de la clase "Con rotura": de todas las roturas reales, qué proporción es capaz de anticipar el modelo.*(Completar esta celda con la lectura de los resultados obtenidos tras la ejecución, señalando qué modelo ofrece el mejor equilibrio entre Precision y Recall, y qué variables resultan más relevantes según el análisis de importancia del apartado 6.)*